# Example analysis: spatial clustering and city network centrality

This notebook demonstrates two example analyses using the cleaned dataset:
1. Global Moran's I on city-level AI filing counts
2. City-level network centrality from the app-model edge list

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from math import radians, sin, cos, asin, sqrt
from pathlib import Path

ROOT = Path('..')
pd.set_option('display.max_rows', 20)

## 1. Load processed data

In [ ]:
reg = pd.read_csv(ROOT / 'data' / 'processed' / '01_filing_registry.csv')
edges = pd.read_csv(ROOT / 'data' / 'processed' / '03_app_model_edges.csv')
print(f'Filing registry: {len(reg)} rows')
print(f'Edge list: {len(edges)} rows')

## 2. City-level filing counts

In [ ]:
city_counts = reg.groupby('city_en').size().reset_index(name='filings')
city_counts = city_counts.sort_values('filings', ascending=False)
print('Top 10 cities by filing count:')
city_counts.head(10)

## 3. City network centrality

Build a directed city-level network from app_city -> model_city, excluding self-loops.

In [ ]:
G = nx.DiGraph()
for _, r in edges.iterrows():
    a, c = r['app_city_cn'], r['model_city_cn']
    if pd.notna(a) and pd.notna(c) and a != c:
        G.add_edge(c, a)  # model city -> app city

G.remove_edges_from(nx.selfloop_edges(G))
print(f'City-level network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

centrality = pd.DataFrame({
    'city': list(G.nodes()),
    'degree': [G.degree(n) for n in G.nodes()],
    'pagerank': [nx.pagerank(G)[n] for n in G.nodes()]
}).sort_values('degree', ascending=False)
centrality.head(10)

## 4. Core-periphery structure (k-core decomposition)

## 5. Spatial autocorrelation: Global Moran's I

Using a k-nearest-neighbor spatial weights matrix (k=5) on log-transformed filing counts.

In [ ]:
## 5. Spatial autocorrelation: Global Moran's I
# Load full city centroid panel (297 cities, including zero-filing cities)
coords = pd.read_csv(ROOT / 'data' / 'raw' / 'city_centroids.csv')
city_counts = reg.groupby('city_en').size().reset_index(name='filings')
panel = coords.merge(city_counts, on='city_en', how='left').fillna({'filings': 0})
# Exclude Sansha (no resident economic population)
panel = panel[panel['city_en'] != 'Sansha'].copy()
print(f'Panel: {len(panel)} cities, {(panel["filings"]>0).sum()} with filings')

from libpysal.weights import KNN
from esda.moran import Moran
w = KNN(panel[['lon','lat']].values, k=5)
w.transform = 'r'
y = np.log1p(panel['filings'].values)
moran = Moran(y, w, permutations=999)
print(f"Moran's I = {moran.I:.3f}, permutation p = {moran.p_sim:.3f}")

In [ ]:
core_num = nx.core_number(G.to_undirected())
kmax = max(core_num.values())
print(f'Max k-core: {kmax}-core')
print('Core cities:', sorted([n for n, c in core_num.items() if c == kmax]))

## Notes

- `data/raw/city_centroids.csv` contains 297 Chinese prefecture-level city centroid coordinates (publicly available).
- The Moran's I analysis uses the full 296-city panel (Sansha excluded), not just cities with filings.
- The edge list includes self-loops (in-house apps); filter them out for cross-city network analysis.
- For LISA cluster maps, use `esda.Moran_Local` and merge results with a shapefile.